In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import pandas as pd

df = pd.read_csv("data/SPX_data_26.02.24.csv")


In [ ]:
df.head(5)

Volatility smile & term structure

In [ ]:
from src.black_scholes import implied_volatility

df['C_mkt'] = (df['bid'] + df['ask']) / 2
df['S_0'] = (df['underlying_bid'] + df['underlying_ask']) / 2
r_annual = 0  # Assunzione tasso d'interesse (4.5%)

df_calls = df

# 3. Calcolo dell'IV su tutto il dataset
df_calls['implied_vol'] = df_calls.apply(
    lambda row: implied_volatility(row['C_mkt'], row['S_0'], row['strike'], row['time_to_maturity'], r_annual),
    axis=1
)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Applica prima i filtri di pulizia (bid > 0, ask > 0, spread sensato)
df_clean = df_calls[(df_calls['bid'] > 0) & (df_calls['ask'] > 0)].copy()
df_clean['IV'] = df_clean['implied_vol'] # la colonna calcolata
df_clean = df_clean.dropna(subset=['IV'])

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Creiamo la superficie tridimensionale
surf = ax.plot_trisurf(df_clean['strike'], df_clean['time_to_maturity'], df_clean['IV'], 
                       cmap='viridis', edgecolor='none', alpha=0.9)

ax.set_xlabel('Strike Price (K)')
ax.set_ylabel('Time to Maturity (T in anni)')
ax.set_zlabel('Volatilità Implicita ($\sigma^{IV}$)')
ax.set_title('Superficie di Volatilità Implicita dello S&P 500')

# Aggiunge la barra dei colori
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
ax.view_init(elev=20, azim=134) # Ruota per vedere bene lo skew

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

# Prendiamo le scadenze uniche disponibili e ne scegliamo alcune significative
scadenze_uniche = sorted(df_clean['expiration'].unique())

for scadenza in scadenze_uniche[:4]: # Mostriamo le prime 4 scadenze come esempio
    df_sub = df_clean[df_clean['expiration'] == scadenza].sort_values(by='strike')
    
    # Visualizziamo solo la zona centrale (ATM) per evitare rumore dagli estremi
    df_sub = df_sub[(df_sub['strike'] > 4500) & (df_sub['strike'] < 7500)]
    
    plt.plot(df_sub['strike'], df_sub['IV'], label=f'Scadenza: {scadenza}', marker='o', markersize=3)

plt.title('S&P 500 Volatility Skew per diverse scadenze')
plt.xlabel('Strike Price (K)')
plt.ylabel('Volatilità Implicita ($\sigma^{IV}$)')
plt.grid(True)
plt.legend()
plt.show()

### Part 2: Heston model

In [ ]:
import numpy as np
import pandas as pd
import src.heston as hst
from tqdm import tqdm  # <--- IMPORTA QUESTO

N_SAMPLES = 20000
S0_mercato = 6852.66  
tasso_risk_free = 0.045

dataset_rows = []

print("Generazione del dataset sintetico in corso...")

# Avvolgi il range dentro tqdm()
for i in tqdm(range(N_SAMPLES), desc="Progresso Integrali"):
    kappa_casuale = np.random.uniform(0.1, 5.0)
    theta_casuale = np.random.uniform(0.01, 0.25)
    xi_casuale = np.random.uniform(0.05, 1.0)
    rho_casuale = np.random.uniform(-0.95, 0.0)
    V0_casuale = np.random.uniform(0.01, 0.25)
    
    maturity_casuale = np.random.uniform(0.1, 3.0) # Partiamo da 0.1 per stabilità
    moneyness = np.random.uniform(0.8, 1.2)
    strike_casuale = S0_mercato * moneyness
    
    try:
        prezzo_sintetico = hst.heston_call_price(
            S0=S0_mercato, 
            K=strike_casuale, 
            T=maturity_casuale, 
            r=tasso_risk_free, 
            kappa=kappa_casuale, 
            theta=theta_casuale, 
            xi=xi_casuale, 
            rho=rho_casuale, 
            V0=V0_casuale
        )
        dataset_rows.append([kappa_casuale, theta_casuale, xi_casuale, rho_casuale, V0_casuale, strike_casuale, maturity_casuale, prezzo_sintetico])
    except:
        continue

columns = ['kappa', 'theta', 'xi', 'rho', 'V0', 'strike', 'time_to_maturity', 'target_price']
df_sintetico = pd.DataFrame(dataset_rows, columns=columns)
print(f"\nGenerazione completata! Righe totali: {len(df_sintetico)}")

In [ ]:
df_sintetico.to_csv('data/heston_synthetic_dataset.csv', index=False)

print(f"\nGenerazione completata e salvata con successo in 'data/heston_synthetic_dataset.csv'! Righe totali: {len(df_sintetico)}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq

def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return np.maximum(0.0, S - K * np.exp(-r * T))
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def calculate_iv_row(row, S, r):
    K = row['strike']
    T = row['time_to_maturity']
    target = row['target_price']
    
    intrinsic_value = max(0.0, S - K * np.exp(-r * T))
    if target <= intrinsic_value or target >= S:
        return np.nan
        
    obj_fun = lambda sigma: bs_call_price(S, K, T, r, sigma) - target
    try:
        return brentq(obj_fun, 1e-5, 5.0)
    except ValueError:
        return np.nan

def main():
    S0_mercato = 6852.66
    tasso_risk_free = 0.045
    
    # Sostituire con il percorso reale del proprio file
    csv_filename = 'data/heston_synthetic_dataset.csv' 
    
    try:
        df = pd.read_csv(csv_filename)
    except FileNotFoundError:
        print(f"Errore: il file '{csv_filename}' non è stato trovato.")
        return

    # 1. Calcolo preliminare della Implied Volatility
    df['implied_vol'] = df.apply(lambda row: calculate_iv_row(row, S0_mercato, tasso_risk_free), axis=1)
    df_clean = df.dropna(subset=['implied_vol'])
    
if __name__ == "main":
    main()

In [ ]:
df

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

def bs_price_vega(S, K, T, r, sigma, option_type):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    vega = S * np.sqrt(T) * norm.pdf(d1)
    
    if option_type == 'C':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
        
    return price, vega

def implied_volatility(target_price, S, K, T, r, option_type, max_iter=100, tol=1e-6):
    sigma = 0.20  # inizializzazione standard al 20%
    for _ in range(max_iter):
        price, vega = bs_price_vega(S, K, T, r, sigma, option_type)
        diff = price - target_price
        if abs(diff) < tol:
            return sigma
        if vega < 1e-6:
            break
        sigma -= diff / vega
    return np.nan

# Caricamento e preparazione dati classici
df = pd.read_csv('data/SPX_data_26.02.24.csv')
df['S'] = (df['underlying_bid'] + df['underlying_ask']) / 2
df['P_mkt'] = (df['bid'] + df['ask']) / 2

r = 0.045  # tasso risk-free costante

df['IV'] = df.apply(
    lambda row: implied_volatility(
        target_price=row['P_mkt'],
        S=row['S'],
        K=row['strike'],
        T=row['time_to_maturity'],
        r=r,
        option_type=row['option_type']
    ), axis=1
)

print(df[['expiration', 'strike', 'option_type', 'IV']].dropna().head())

In [ ]:
df.to_csv('data/SPX_data_with_IV.csv', index=False)

In [ ]:
import pandas as pd

# Caricamento del dataset con le IV
df_iv = pd.read_csv('data/SPX_data_with_IV.csv')

# Rimozione delle righe dove la colonna IV è NaN
df_clean = df_iv.dropna(subset=['IV'])

# Salvataggio del dataset pulito
df_clean.to_csv('data/SPX_data_clean_IV.csv', index=False)

# Visualizzazione delle prime righe e del conteggio residuo
print(f"Righe rimosse: {len(df_iv) - len(df_clean)}")
print(f"Righe rimanenti: {len(df_clean)}")
print(df_clean[['expiration', 'strike', 'option_type', 'IV']].head())